# 🔢 Notebook 02: Embedding & Indexing

Notebook này thực hiện:
1. Load processed chunks
2. Embedding với PhoBERT và multilingual-e5
3. Index vào ChromaDB
4. Test retrieval

In [ ]:
import os, sys
PROJECT_DIR = '/content/vietnamese-legal-qa'  # Colab
# PROJECT_DIR = '.'  # Local
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from src.config.settings import Settings
from src.services.data_ingestion_service import DataIngestionService

settings = Settings.load('config.yaml')
ingestion = DataIngestionService(settings)
print('Ready!')

In [ ]:
# === Embedding với multilingual-e5-large ===
stats_e5 = ingestion.ingest_from_files(
    directory='data/raw',
    embedding_model='multilingual_e5',
    collection_name='legal_docs_e5'
)
print(f'\n✅ E5 indexing complete: {stats_e5}')

In [ ]:
# === Embedding với PhoBERT ===
stats_phobert = ingestion.ingest_from_files(
    directory='data/raw',
    embedding_model='phobert',
    collection_name='legal_docs_phobert'
)
print(f'\n✅ PhoBERT indexing complete: {stats_phobert}')

In [ ]:
# === Test retrieval ===
from src.components.embedding_engine import EmbeddingEngine
from src.components.vector_store import VectorStoreManager

emb_engine = EmbeddingEngine(settings)
vector_store = VectorStoreManager(settings)

# Test query
test_query = "Thời hạn hợp đồng lao động xác định là bao lâu?"
query_vec = emb_engine.embed_query(test_query, 'multilingual_e5')

results = vector_store.similarity_search(
    query_embedding=query_vec.tolist(),
    collection_name='legal_docs_e5',
    top_k=5
)

print(f'Query: {test_query}')
print(f'\nTop {len(results)} results:')
for i, r in enumerate(results, 1):
    print(f'  {i}. [Score: {r.similarity_score:.4f}] {r.content[:100]}...')

print('\n✅ Embedding & Indexing complete!')